# Speech Analysis - CNN-LSTM Model Training

This notebook trains a 1D-CNN + LSTM hybrid model to classify speech patterns
as **Normal** or **At-Risk** for dyslexia screening.

## Dataset
Place audio files in `../data/speech/` with subdirectories: `Normal/`, `AtRisk/`

Suggested sources:
- LibriSpeech (normal baseline)
- TORGO (disordered speech)
- Custom recordings

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras

import sys
sys.path.insert(0, os.path.abspath('..'))

from app.ml.speech.audio_processor import load_audio
from app.ml.speech.feature_extractor import extract_mfcc_features
from app.ml.speech.speech_model import build_speech_model

print(f'TensorFlow version: {tf.__version__}')

In [ ]:
# Configuration
DATA_DIR = '../data/speech'
CLASS_NAMES = ['Normal', 'AtRisk']
BATCH_SIZE = 32
EPOCHS = 30
MODEL_SAVE_PATH = '../saved_models/speech_model.keras'

In [ ]:
# Load and extract features from audio files
features = []
labels = []

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(DATA_DIR, class_name)
    if not os.path.exists(class_dir):
        print(f'WARNING: {class_dir} not found!')
        continue
    
    files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.wav', '.mp3', '.flac'))]
    print(f'{class_name}: {len(files)} audio files')
    
    for fname in files:
        fpath = os.path.join(class_dir, fname)
        try:
            signal, sr = load_audio(fpath)
            mfcc = extract_mfcc_features(signal, sr)
            features.append(mfcc)
            labels.append(class_idx)
        except Exception as e:
            print(f'Error processing {fpath}: {e}')

X = np.array(features)
y = keras.utils.to_categorical(labels, num_classes=len(CLASS_NAMES))
print(f'\nTotal: {len(X)} samples, shape: {X.shape}')

In [ ]:
# Train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print(f'Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}')

In [ ]:
# Build and compile model
model = build_speech_model(input_shape=(200, 39), num_classes=len(CLASS_NAMES))
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Train
callbacks = [
    keras.callbacks.EarlyStopping(patience=7, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
)

In [ ]:
# Evaluate
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'\nTest Accuracy: {test_acc:.4f}')
print(f'Test Loss: {test_loss:.4f}')

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train')
ax1.plot(history.history['val_accuracy'], label='Validation')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='Train')
ax2.plot(history.history['val_loss'], label='Validation')
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Save model
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)
model.save(MODEL_SAVE_PATH)
print(f'Model saved to {MODEL_SAVE_PATH}')